# Manual Annotation of Team Attributions for LLM Validation Purposes

In the first part of this notebook, 14 surveyed teams are randomly selected to have their members and task descriptions manually extracted. The manual annotation of members and tasks is done in order to create a dataframe with columns `"RawTextName"`, `"RosterName"`, `"Team"`, and `"TasksDescription"`. This dataframe is then turned into a JSON file, and will be used as a gold standard in order to evaluate the performance of the LLM doing the same name and task extraction from raw attributions texts.

In the second part, 5 out of 6 surveyed teams without extracted raw attributions texts (check `6_match_teams_2022_attributions`) are also manually annotated. Examples from these teams are used for defining the few-shot prompt and the rules for the LLM. During the prompt engineering process, these teams are used as validation that the LLMs perform well. 

## 1. Random Surveyed Teams Selection for Model Evaluation

In [1]:
import pandas as pd
import random
import json
import re
import math

In [2]:
df_survey = pd.read_csv("../data/igem_ties_surveys/2022/survey_tasks.csv")

In [5]:
df_survey['user_id'].nunique()

776

In [5]:
df_team_meta = pd.read_table("../data/igem_scrapping_2025/team_meta_full.tsv")

In [12]:
# Surveyed teams without attributions

teams_without_attributions = ['ITESO_Guadalajara', 'HKU_HongKong', 'Patras_Medicine', 'NJU-China', 'Stanford', 'Vilnius-Lithuania' ]

In [13]:
# Randomly select 14 unique teams from the survey data

random.seed(42)  
filtered_teams = [team for team in df_survey["team"].dropna().unique() if team not in teams_without_attributions]
random_teams = random.sample(filtered_teams, min(14, len(filtered_teams)))

In [14]:
random_teams

['Technion-Israel',
 'CSMU_Taiwan',
 'Aalto-Helsinki',
 'UPNAvarra_Spain',
 'ICT-Mumbai',
 'Goettingen',
 'Freiburg',
 'Cambridge',
 'CPU_Nanjing',
 'UCBerkeley',
 'Sogang_Korea',
 'BostonU_HW',
 'TU_Braunschweig',
 'Montpellier']

I forgot to add a condition for only accepted teams. As UCBerkeley was withdrawn, it was not used in the manual annotation. Therefore, we will randomly select one more surveyed team so we could have 14 teams for model evaluation.

In [10]:
# Randomly select 1 more team from the survey data

random.seed(43)

accepted_teams = df_team_meta[df_team_meta["Status"] == "accepted"]["Team"].unique()

filtered_accepted_teams = [
    team for team in df_survey["team"].dropna().unique()
    if team in accepted_teams and team not in teams_without_attributions
]

random_accepted_team = random.sample(filtered_accepted_teams, min(1, len(filtered_accepted_teams)))

In [11]:
random_accepted_team

['Aboa']

In [15]:
random_teams.append(random_accepted_team[0])

In [16]:
random_teams

['Technion-Israel',
 'CSMU_Taiwan',
 'Aalto-Helsinki',
 'UPNAvarra_Spain',
 'ICT-Mumbai',
 'Goettingen',
 'Freiburg',
 'Cambridge',
 'CPU_Nanjing',
 'UCBerkeley',
 'Sogang_Korea',
 'BostonU_HW',
 'TU_Braunschweig',
 'Montpellier',
 'Aboa']

## 2. Member Names and Raw Task Descriptions

### 2.1 Test Teams

In [17]:
def load_attributions_json(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

filepath = "../data/attributions/2022_attributions/raw_attributions_2022.json"

data = load_attributions_json(filepath)

In [18]:
random_teams_attributions_dict = [entry for entry in data if entry["teamName"] in random_teams]

In [19]:
# Save the random teams' raw attributions to a new JSON file
output_path = "../data/attributions/2022_attributions/random_teams_raw_attributions_2022.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(random_teams_attributions_dict, f, ensure_ascii=False, indent=2)

In [20]:
df_roster = pd.read_table("../data/attributions/2022_attributions/team_rosters_2022.tsv")

In [ ]:
# Randomly selected teams' rosters

roster_random_teams_dict = (
    df_roster[df_roster["Team"].isin(random_teams)]
    .groupby("Team")["FullName"]
    .apply(list)
    .to_dict()
)

In [22]:
roster_random_teams_dict

{'Aalto-Helsinki': ['Ilse Kaaja',
  'Juuso Taskinen',
  'Rishi Banerjee',
  'Sami Jalil',
  'A. Sesilja Aranko',
  'Ville Paavilainen',
  'Heli Viskari',
  'Markus Linder',
  'Melissa Hendr�n',
  'Veera Kurki',
  'Amna Gul',
  'Anniina Könönen',
  'Diogo Dias',
  'Hanna Nebelung',
  'Joose Lankia',
  'Lilith Heiland',
  'Mari Keskiivari',
  'Zsofia Hesketh'],
 'Aboa': ['Pauli Kallio',
  'Juuli Hietarinne',
  'Malin Eriksson',
  'Anni Marjomaa',
  'Evelina Ojaniittu',
  'Iida Raaska',
  'Jesper Mickos',
  'Jonna Pohjankukka',
  'Kevät Sova',
  'Kristiina Keski-Oja',
  'Minttu Turunen',
  'Nelli Heiskanen',
  'Otto Rydman',
  'Sini Kylä-Kaila',
  'Sofia Antin'],
 'BostonU_HW': ['Diana Arguijo',
  'Hailey Lenn Gordon',
  'Douglas Densmore',
  'Alexander Barutis',
  'Aya Kassem',
  'Julia Nowak',
  'Stephen Sweet',
  'zakir kadwa'],
 'CPU_Nanjing': ['Jingzhi Hu',
  'Hai Qian',
  'Xin Wang',
  'Mayu Shan',
  'Chang Liu',
  'Chengjun Yuan',
  'Haotian Liu',
  'Qiuhan Ren',
  'Ruochen Qian',


In the manually annotated dataset (made in Google Sheets), the `"RawTextName"` represents every member name that is found in the team's raw attributions text, but also exists in the official team's roster (external and other contributors are not considered here). The names from the attributions and the roster sometimes differ in spelling. The `"RosterName"` columns contains all names found in the team's roster. If a variant of the roster name is not found in the team's attribution text, values in columns `"RawTextName"` and `"TasksDescriptions"` are left null. Fourteen randomly selected teams are considered, with their names in the `"Team"` column. `"TasksDescription"` includes full text describing tasks of one person. The value for the `"TasksDescription"` column can be the actual tasks description and/or the team-defined task categories, whatever the team has written for each member's name. Newlines "\n" and extra blank spaces will later be removed from the descriptions. `"Comments"` column describes edge cases found in each team  (e.g., some teams include sentences that apply to all members; these sentences should be added to each individual member even if their name is not explicitly mentioned).

In [ ]:
# Load manually annotated tsv attributions

df_manual_random_attributions = pd.read_csv("../data/attributions/2022_attributions/manual_annotation/14_random_teams_tasks.tsv", sep="\t")

In [4]:
df_manual_random_attributions

,RawTextName,RosterName,Team,TasksDescription,Comments
0,Melissa Hendrén,Melissa Hendr�n,Aalto-Helsinki,is one of the team co-leads to the Aalto-Helsi...,The team does not include a name subtitle for ...
1,Veera Kurki,Veera Kurki,Aalto-Helsinki,is one of the team co-leads to the Aalto-Helsi...,NaN
2,Diogo Dias,Diogo Dias,Aalto-Helsinki,is head of the dry-lab. He is involved in the ...,NaN
3,Amna Gul,Amna Gul,Aalto-Helsinki,is an active member of the wet-lab team. She c...,NaN
4,Lilith Heiland,Lilith Heiland,Aalto-Helsinki,is head of the writing team and part of the we...,NaN
...,...,...,...,...,...
236,Minttu Turunen,Minttu Turunen,Aboa,All team members participated in the general a...,NaN
237,Nelli Heiskanen,Nelli Heiskanen,Aboa,All team members participated in the general a...,NaN
238,Otto Rydman,Otto Rydman,Aboa,All team members participated in the general a...,NaN
239,Sini Kylä-Kaila,Sini Kylä-Kaila,Aboa,All team members participated in the general a...,NaN


In [ ]:
# Convert to JSON format, to be comparable with model outputs
# List of objects where each object has a team name as a key, and the value is a list of members with their names and task descriptions

def convert_df_to_json_format(df):
    if 'Comments' in df.columns:
        df = df.drop(columns=['Comments'])

    json_data = []
    for team, group in df.groupby('Team'):
        members = group[['RosterName', 'RawTextName', 'TasksDescription']].to_dict(orient='records')
        json_data.append({team: members})

    return json_data

In [5]:
json_test_data = convert_df_to_json_format(df_manual_random_attributions)
json_test_data

[{'Aalto-Helsinki': [{'RosterName': 'Melissa Hendr�n',
    'RawTextName': 'Melissa Hendrén',
    'TasksDescription': 'is one of the team co-leads to the Aalto-Helsinki team. She is head of the wet-lab team and involved in fundraising, collaborations with other teams, writing, synbio and travelling organisation.'},
   {'RosterName': 'Veera Kurki',
    'RawTextName': 'Veera Kurki',
    'TasksDescription': 'is one of the team co-leads to the Aalto-Helsinki team. She is head of the human practices, the social media team and chair of synbio. She is part of the wet-lab and involved in fundraising, collaborations, writing, social media and travelling organisation.'},
   {'RosterName': 'Diogo Dias',
    'RawTextName': 'Diogo Dias',
    'TasksDescription': 'is head of the dry-lab. He is involved in the data analysis of the project, blogpost writing and review as well as participating in the development of the promotion video.'},
   {'RosterName': 'Amna Gul',
    'RawTextName': 'Amna Gul',
    '

In [6]:
# Remove \n, \\n, and multiple spaces between words from the attributions text
# Convert NaN values to None (supported by the JSON format)

def clean_json_data(json_data):

    for team in json_data:
        for team_name, members in team.items():
            for member in members:
                # Normalize TasksDescription
                tasks = member.get("TasksDescription")
                if isinstance(tasks, str):
                    cleaned = tasks.replace("\\n", " ").replace("\n", " ")
                    cleaned = re.sub(r"\s+", " ", cleaned)
                    member["TasksDescription"] = cleaned.strip()
                
                # Convert any NaN value in the member dict to None
                for key, value in member.items():
                    if isinstance(value, float) and math.isnan(value):
                        member[key] = None
    return json_data


In [8]:
json_test_data = clean_json_data(json_test_data)
json_test_data

[{'Aalto-Helsinki': [{'RosterName': 'Melissa Hendr�n',
    'RawTextName': 'Melissa Hendrén',
    'TasksDescription': 'is one of the team co-leads to the Aalto-Helsinki team. She is head of the wet-lab team and involved in fundraising, collaborations with other teams, writing, synbio and travelling organisation.'},
   {'RosterName': 'Veera Kurki',
    'RawTextName': 'Veera Kurki',
    'TasksDescription': 'is one of the team co-leads to the Aalto-Helsinki team. She is head of the human practices, the social media team and chair of synbio. She is part of the wet-lab and involved in fundraising, collaborations, writing, social media and travelling organisation.'},
   {'RosterName': 'Diogo Dias',
    'RawTextName': 'Diogo Dias',
    'TasksDescription': 'is head of the dry-lab. He is involved in the data analysis of the project, blogpost writing and review as well as participating in the development of the promotion video.'},
   {'RosterName': 'Amna Gul',
    'RawTextName': 'Amna Gul',
    '

In [9]:
with open("../data/attributions/2022_attributions/manual_annotation/14_test_teams.json", 'w', encoding='utf-8') as f:
    json.dump(json_test_data, f, ensure_ascii=False, indent=2)

### 2.2 Validation (Prompt Engineering) Teams

To refine our prompt, we selected 6 surveyed teams that either do not have an iGEM website folder or are missing their attributions `.txt` or `.html` file (check out `6_match_teams_2022_attributions.ipynb` for details). Their task attributions were collected manually from their GitLab repositories. The reason they were not previously included in the extraction of attribution texts alongside other teams is that their attributions were mostly written in JavaScript and not HTML files. However, one team is removed because it did not list any performed tasks on its Attributions webpage, so there are 5 validation teams left.

In [11]:
# Load manually annotated tsv attributions
# One team is also present in the 14 random teams because it was noticed that its attributions were written in an unreadable table
# For this team, the attributions were manually extracted from the iGEM wiki page
# The team is dropped from the validation test and used only for the random teams evaluation

df_manual_validation_attributions = pd.read_csv("../data/attributions/2022_attributions/manual_annotation/7_teams_members_descriptions_from_wikis.tsv", sep="\t")

df_manual_validation_attributions = df_manual_validation_attributions[
    ~df_manual_validation_attributions["Team"].isin(["Sogang_Korea", "HKU-HongKong"])
]

In [47]:
df_manual_validation_attributions

,RawTextName,RosterName,Team,TasksDescription,Comments
21,Óscar Ariel Rojas Rejón,Oscar Rojas-Rejon,ITESO_Guadalajara,for his constant support throughout the compet...,NaN
22,Sarah Ratkovich Gonzalez,Sarah Ratkovich-Gonzalez,ITESO_Guadalajara,for her unconditional support as our Principal...,NaN
23,Cristóbal Camarena Bernard,Cristobal Camarena,ITESO_Guadalajara,"for being our contact with the university, for...",NaN
24,Claudia Denisse Valdés Michel,Claudia Valdes,ITESO_Guadalajara,For her amazing work as the team leader and as...,NaN
25,Sofía Carolina Bárcena Pérez,Sofia Carolina Barcena Perez,ITESO_Guadalajara,For her outstanding work as responsible for La...,NaN
...,...,...,...,...,...
125,Gabrielė,Gabriele Olendraite,Vilnius-Lithuania,Protein expression optimisation and purificati...,NaN
126,Laura,Laura Milaknytė,Vilnius-Lithuania,Hardware. The 6th SynBio Sense. Baltic Jambore...,NaN
127,Simona,Simona Bendziute,Vilnius-Lithuania,National Television LRT. Marketing. Baltic Jam...,NaN
128,Simonija,Simonija Kelpsaite,Vilnius-Lithuania,Plasmid vector contruction. Peptide affinity e...,NaN


In [12]:
# Convert to JSON format as well

json_validation_data = convert_df_to_json_format(df_manual_validation_attributions)

In [13]:
# Clean the JSON data
json_validation_data = clean_json_data(json_validation_data)

In [15]:
with open("../data/attributions/2022_attributions/manual_annotation/5_validation_teams.json", 'w', encoding='utf-8') as f:
    json.dump(json_validation_data, f, ensure_ascii=False, indent=2)